# Week 2 - Data pipeline

## Summary

_This assignment introduces the first stage of a machine learning project for a
city-wide bike sharing company. The goal is not yet to build a forecasting
model, but to prepare the data that will support later machine learning work.
You will combine data from multiple sources, explore and transform it, and
organize the result as a reusable preprocessing pipeline. In doing so, you will
gain experience with core data engineering and machine learning preparation
tasks, using tools such as pandas, Jupyter notebooks, and Dagster._


## Content:
1) Phase 1: Data Ingestion and Exploratory Data Analysis (EDA)


## Phase 1: EDA

Here we establish a baseline understanding of our raw datasets. The following sequences utilize custom exploration modules to extract schema summaries, verify standard data types, and quantify null values across our core datasets.

In [15]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
import seaborn as sns

In [16]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

sns.set_theme(style="whitegrid")

In [17]:
DATA_DIR = Path("../mlops/week-2/data")
PATH_DIRECT_PICKUP_BIKE = DATA_DIR / "direct_pickup_bike_rentals.csv"
PATH_HOLIDAYS = DATA_DIR / "holidays.csv"
PATH_REGISTERED_BIKE = DATA_DIR / "registered_bike_rentals.csv"
PATH_WEATHER = DATA_DIR / "weather.csv"

In [18]:
import explore

In [19]:
explore.explore_csv(PATH_HOLIDAYS)
"""Explores a CSV file by printing its head, info, and null value counts."""

   id        date                                 holiday
0   1  2011-01-17  Dr. Martin Luther King, Jr.'s Birthday
1   2  2011-02-21                   Washington's Birthday
2   3  2011-04-15        D.C. Emancipation Day (observed)
3   4  2011-05-30                            Memorial Day
4   5  2011-07-04                        Independence Day
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   id       21 non-null     int64 
 1   date     21 non-null     object
 2   holiday  21 non-null     object
dtypes: int64(1), object(2)
memory usage: 632.0+ bytes
None
id         0
date       0
holiday    0
dtype: int64


'Explores a CSV file by printing its head, info, and null value counts.'

### Intermediate observation 1: Holidays
---
* *Volume & Completeness: There are 21 entries with exactly 0 missing values across all columns.*
* *Temporal Format: The `date` column is currently stored as a generic object (text string).*
* *Granularity: The timestamps are precise to the exact day (e.g., 2011-01-17).*
* *Categorical/Relational Data: The `holiday` column contains the names of the holidays as text strings, and `id` is stored as an integer (int64).*
---

#### Potential action:
* convert `date` column to an appropriate datetime datatype
* convert holiday column to the true/false marker for the date
* id may be irrelevant here

In [20]:
explore.explore_csv(PATH_REGISTERED_BIKE)

   id             datetime  user_id  location_id
0   1  2011-01-01 00:05:09      158           16
1   2  2011-01-01 00:05:21      262           18
2   3  2011-01-01 00:05:39       68           18
3   4  2011-01-01 00:12:05       12            9
4   5  2011-01-01 00:25:58       91           11
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2672662 entries, 0 to 2672661
Data columns (total 4 columns):
 #   Column       Dtype 
---  ------       ----- 
 0   id           int64 
 1   datetime     object
 2   user_id      int64 
 3   location_id  int64 
dtypes: int64(3), object(1)
memory usage: 81.6+ MB
None
id             0
datetime       0
user_id        0
location_id    0
dtype: int64


,id,datetime,user_id,location_id
0,1,2011-01-01 00:05:09,158,16
1,2,2011-01-01 00:05:21,262,18
2,3,2011-01-01 00:05:39,68,18
3,4,2011-01-01 00:12:05,12,9
4,5,2011-01-01 00:25:58,91,11
...,...,...,...,...
2672657,2672658,2012-12-31 23:53:45,140,1
2672658,2672659,2012-12-31 23:54:01,191,17
2672659,2672660,2012-12-31 23:55:03,283,5
2672660,2672661,2012-12-31 23:55:41,137,16


### Intermediate observation 2: Regestered Bike Rentals
---
* *Volume & Completeness: There are 2,672,662 entries with exactly 0 missing values across all columns.*
* *Temporal Format: The `datetime` column is currently stored as a generic object (text string).*
* *Granularity: The timestamps are highly precise, down to the exact second (e.g., 2011-01-01 00:05:09).*
* *Categorical/Relational Data: `user_id` and `location_id` are stored as integers (int64), acting as relational keys to other potential tables. The `id` column is a simple incremental index.*
---
*(Pending execution output - expecting similar structure to Registered Rentals)*

#### Potential action:
* convert `datetime` column to an appropriate datetime datatype
* extract the pure date from `datetime` into a new column 
* drop the `id` column 
* **Integration Step:** Add a boolean flag (e.g., `is_registered = False`) and concatenate this vertically with the Registered Bike Rentals dataset to create a single master transactions table.

In [21]:
explore.explore_csv(PATH_DIRECT_PICKUP_BIKE)

   id             datetime  user_id  location_id
0   1  2011-01-01 00:24:04      232            2
1   2  2011-01-01 00:30:19       54           14
2   3  2011-01-01 00:39:08      201            5
3   4  2011-01-01 01:01:12      298           13
4   5  2011-01-01 01:02:37       23           14
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 620017 entries, 0 to 620016
Data columns (total 4 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   id           620017 non-null  int64 
 1   datetime     620017 non-null  object
 2   user_id      620017 non-null  int64 
 3   location_id  620017 non-null  int64 
dtypes: int64(3), object(1)
memory usage: 18.9+ MB
None
id             0
datetime       0
user_id        0
location_id    0
dtype: int64


,id,datetime,user_id,location_id
0,1,2011-01-01 00:24:04,232,2
1,2,2011-01-01 00:30:19,54,14
2,3,2011-01-01 00:39:08,201,5
3,4,2011-01-01 01:01:12,298,13
4,5,2011-01-01 01:02:37,23,14
...,...,...,...,...
620012,620013,2012-12-31 23:30:18,156,6
620013,620014,2012-12-31 23:37:05,119,4
620014,620015,2012-12-31 23:42:35,31,18
620015,620016,2012-12-31 23:44:48,23,5


### Intermediate observation 3: Direct Pickup Bike Rentals
---
* *Volume & Completeness: There are 620,017 entries with exactly 0 missing values across all columns.*
* *Temporal Format: The `datetime` column is currently stored as a generic object (text string).*
* *Granularity: The timestamps are highly precise, down to the exact second (e.g., 2011-01-01 00:24:04).*
* *Categorical/Relational Data: The column structure perfectly matches the Registered Bike Rentals dataset. `user_id` and `location_id` are integers (int64), and `id` is a simple incremental index.*
---

#### Potential action:
* add a boolean flag (e.g., `is_registered = False`) to distinguish these rows
* concatenate vertically with the Registered Bike Rentals dataset to create a single master transactions table
* apply the previous transformations (convert `datetime`, extract pure `date`, drop `id`) to the unified master table

In [22]:
explore.explore_csv(PATH_WEATHER)

   id             datetime conditions  temperature_c  perceived_temperature_c  humidity  windspeed_kmh
0   1  2011-01-01 00:00:00      clear            3.3                      3.0      81.0            0.0
1   2  2011-01-01 01:00:00      clear            2.3                      2.0      80.0            0.0
2   3  2011-01-01 02:00:00      clear            2.3                      2.0      80.0            0.0
3   4  2011-01-01 03:00:00      clear            3.3                      3.0      75.0            0.0
4   5  2011-01-01 04:00:00      clear            3.3                      3.0      75.0            0.0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17379 entries, 0 to 17378
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       17379 non-null  int64  
 1   datetime                 17379 non-null  object 
 2   conditions               17379 non-null  object 
 3  

,id,datetime,conditions,temperature_c,perceived_temperature_c,humidity,windspeed_kmh
0,1,2011-01-01 00:00:00,clear,3.3,3.0,81.0,0.0
1,2,2011-01-01 01:00:00,clear,2.3,2.0,80.0,0.0
2,3,2011-01-01 02:00:00,clear,2.3,2.0,80.0,0.0
3,4,2011-01-01 03:00:00,clear,3.3,3.0,75.0,0.0
4,5,2011-01-01 04:00:00,clear,3.3,3.0,75.0,0.0
...,...,...,...,...,...,...,...
17374,17375,2012-12-31 19:00:00,clouds,4.2,1.0,60.0,11.0
17375,17376,2012-12-31 20:00:00,clouds,4.2,1.0,60.0,11.0
17376,17377,2012-12-31 21:00:00,clear,4.2,1.0,60.0,11.0
17377,17378,2012-12-31 22:00:00,clear,4.2,2.0,56.0,9.0


### Intermediate observation 4: Weather
---
* *Volume & Completeness: There are 17,379 entries with exactly 0 missing values across all columns.*
* *Temporal Format: The `datetime` column is currently stored as a generic object (text string).*
* *Granularity: The timestamps are recorded at an exact hourly cadence (e.g., 2011-01-01 00:00:00).*
* *Features: Meteorological data (temperature, humidity, windspeed) are properly stored as continuous numbers (floats), and `conditions` is a categorical string. The `id` column is a simple incremental index.*
---

#### Potential action:
* convert `datetime` column to an appropriate datetime datatype
* drop the `id` column as it is an irrelevant index for our pipeline
* when merging later, we will need to round the highly precise Bike Rentals timestamps to the nearest hour to successfully align with this dataset

## Phase 2: Data Transformation Pipeline

In this phase, we transition from exploration to active data engineering. To build a robust foundation for our future machine learning pipeline, we will structure our transformations as **modular, idempotent functions** rather than global state mutations.

**Architectural Rationale:** Professional pipeline orchestrators (e.g., Dagster) rely on isolated computing tasks. By wrapping our cleaning logic inside dedicated functions for each dataset, we ensure:
1. **Reproducibility:** The transformation behaves identically every time it is called.
2. **Immutability:** We strictly avoid mutating raw data variables in place; instead, we ingest raw sources and return strictly formatted, independent DataFrames.
3. **Testability:** Each source's formatting rules are isolated and can be debugged independently.

**Phase Objectives:**
* **Temporal Casting:** Safely convert all raw string timestamps into Pandas `datetime64[ns]` objects to enable time-series operations.
* **Feature Engineering:** Extract required integration keys (e.g., pure dates) and build categorical markers (e.g., `is_holiday`).
* **Dimensionality Reduction:** Drop arbitrary index columns (`id`) that provide no predictive value to the downstream model.

In [23]:
def transform_holidays(file_path):
    """
    Ingests and transforms the raw holidays dataset for merging.
    """
    raw_df = pd.read_csv(file_path)
    clean_df = (
        raw_df.assign(
            date = pd.to_datetime(raw_df['date']),
            is_holiday = True
        )
        .drop(columns=['id', 'holiday'])
    )
    return clean_df
df_holidays_clean = transform_holidays(PATH_HOLIDAYS)
print(df_holidays_clean.head(3))
print("===")
print(df_holidays_clean.info())

        date  is_holiday
0 2011-01-17        True
1 2011-02-21        True
2 2011-04-15        True
===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   date        21 non-null     datetime64[ns]
 1   is_holiday  21 non-null     bool          
dtypes: bool(1), datetime64[ns](1)
memory usage: 317.0 bytes
None


### Transforming of the weather_dataset
**For the weather dataset our goals are:**
1) Convert the datetime string column into a mathematical time object.
2) Drop the irrelevant id column.

In [24]:
def transform_weather(weather_path):
    raw_weather = pd.read_csv(weather_path)
    raw_weather['datetime'] = pd.to_datetime(raw_weather['datetime'])
    clean_weather = raw_weather.drop(columns=['id'])
    return clean_weather

df_weather_clean = transform_weather(PATH_WEATHER)
print(df_weather_clean.info())
print(df_weather_clean['datetime'].dtype)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17379 entries, 0 to 17378
Data columns (total 6 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   datetime                 17379 non-null  datetime64[ns]
 1   conditions               17379 non-null  object        
 2   temperature_c            17379 non-null  float64       
 3   perceived_temperature_c  17379 non-null  float64       
 4   humidity                 17379 non-null  float64       
 5   windspeed_kmh            17379 non-null  float64       
dtypes: datetime64[ns](1), float64(4), object(1)
memory usage: 814.8+ KB
None
datetime64[ns]


### Transforming of the rentals dataset:

1) Convert the datetime string column into a mathematical time object.
2) Drop the irrelevant id column.
3) Add boolean is_regestered for the future merging.

In [25]:
def transform_registered_rentals(file_path):
    """
    Ingests and transforms the raw bike rentals dataset for merging.
    """
    df = pd.read_csv(file_path)
    df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce')
    df = df.drop(columns=['id'])
    df['is_registered'] = True
    
    return df

# Execute the final pipeline step
df_rentals_clean = transform_registered_rentals(PATH_DIRECT_PICKUP_BIKE)

print(df_rentals_clean.head(3))
print("===")
print(df_rentals_clean.info())

             datetime  user_id  location_id  is_registered
0 2011-01-01 00:24:04      232            2           True
1 2011-01-01 00:30:19       54           14           True
2 2011-01-01 00:39:08      201            5           True
===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 620017 entries, 0 to 620016
Data columns (total 4 columns):
 #   Column         Non-Null Count   Dtype         
---  ------         --------------   -----         
 0   datetime       620017 non-null  datetime64[ns]
 1   user_id        620017 non-null  int64         
 2   location_id    620017 non-null  int64         
 3   is_registered  620017 non-null  bool          
dtypes: bool(1), datetime64[ns](1), int64(2)
memory usage: 14.8 MB
None


### Transforming of the direct_pickups dataset:

1) Convert the datetime string column into a mathematical time object.
2) Drop the irrelevant id column.
3) Add boolean is_regestered for the future merging.

In [26]:
def transform_direct_pickups(file_path):
    """
    Ingests raw direct pickups, formats time, flags the source, and drops indexes.
    """
    df = pd.read_csv(file_path)
    df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce')
    df['is_registered'] = False
    df = df.drop(columns=['id'])
    
    return df

### Phase 2.1 Execution and Validation:

With all four modular functions built, we can now execute check the result and validate boundaries.

In [27]:
print("====Executing Data Transformation Pipeline...====")
df_holidays_clean = transform_holidays(PATH_HOLIDAYS)
df_weather_clean = transform_weather(PATH_WEATHER)
df_registered_clean = transform_registered_rentals(PATH_REGISTERED_BIKE)
df_direct_clean = transform_direct_pickups(PATH_DIRECT_PICKUP_BIKE)

====Executing Data Transformation Pipeline...====


### 2.1.1 Validate conversion success (Check for coerced failures)

In [28]:
print("\n--- Invalid Timestamps Created ---")
print(f"Registered Rentals: {df_registered_clean['datetime'].isna().sum()}")
print(f"Direct Pickups:     {df_direct_clean['datetime'].isna().sum()}")
print(f"Weather:            {df_weather_clean['datetime'].isna().sum()}")
print(f"Holidays:           {df_holidays_clean['date'].isna().sum()}")


--- Invalid Timestamps Created ---
Registered Rentals: 0
Direct Pickups:     0
Weather:            0
Holidays:           0


### 2.1.2 Validate conversion success (Check for coerced failures)

In [29]:
print("\n--- Temporal Boundaries ---")
print(f"Registered: {df_registered_clean['datetime'].min()} -> {df_registered_clean['datetime'].max()}")
print(f"Direct:     {df_direct_clean['datetime'].min()} -> {df_direct_clean['datetime'].max()}")
print(f"Weather:    {df_weather_clean['datetime'].min()} -> {df_weather_clean['datetime'].max()}")
print(f"Holidays:   {df_holidays_clean['date'].min()} -> {df_holidays_clean['date'].max()}")


--- Temporal Boundaries ---
Registered: 2011-01-01 00:05:09 -> 2012-12-31 23:55:53
Direct:     2011-01-01 00:24:04 -> 2012-12-31 23:51:18
Weather:    2011-01-01 00:00:00 -> 2012-12-31 23:00:00
Holidays:   2011-01-17 00:00:00 -> 2012-12-25 00:00:00


### 2.1.3 Gaps identification:

---
We can see that the dataset contains both time and date stamps. Both time stamps should be continuous. For the weather, it is important that the data is recorded hourly, while for the bike rentals, we are interested in whether the records are daily. Let's check this:
---
*In this section, we will check if any data is missing.*

In [30]:
weather_perfect_h = pd.date_range(start=df_weather_clean['datetime'].min(), end=df_weather_clean['datetime'].max(), freq='h')
print("======Missign hours:=======")
weathers_diff = weather_perfect_h.difference(df_weather_clean['datetime'])
print(f"the number of missing hours: {len(weathers_diff)}")

======Missign hours:=======
the number of missing hours: 165


In [31]:
reg_bike_days = pd.to_datetime(df_registered_clean['datetime'].dt.date).unique()
reg_bike_perfect_days = pd.date_range(start=reg_bike_days.min(), end=reg_bike_days.max(), freq='D')
mis_days_rent = reg_bike_perfect_days.difference(reg_bike_days)
print("======Missign days in Reg Bikes:=======")
print(f"the number of missing days in Registered Bike Rentals: {len(mis_days_rent)}")

======Missign days in Reg Bikes:=======
the number of missing days in Registered Bike Rentals: 0


In [32]:
dir_pickup_bike_days = pd.to_datetime(df_direct_clean['datetime'].dt.date).unique()
dir_pickup_bike_perfect_days = pd.date_range(start=reg_bike_days.min(), end=reg_bike_days.max(), freq='D')
mis_days_direct = dir_pickup_bike_perfect_days.difference(dir_pickup_bike_days)
print("======Missign days in Dir Bikes:=======")
print(f"the number of missing days in Registered Bike Rentals: {len(mis_days_direct)}")

======Missign days in Dir Bikes:=======
the number of missing days in Registered Bike Rentals: 0


## Phase 3: Dataset Integration & Merging

With our data cleaned and validated, we must now consolidate our independent sources into a single, unified "Master DataFrame" suitable for machine learning training. 

This phase requires handling two distinct types of data integration:
1. **Vertical Concatenation:** Combining our two identical transaction logs (Registered and Direct Pickups) into a single ledger, utilizing the boolean flags engineered in Phase 2 to preserve the source identity.
2. **Horizontal Relational Joins:** Appending our contextual dimensions (Weather and Holidays) to the transaction ledger. This requires careful **temporal alignment**:
    * *Hourly Alignment:* Truncating/rounding transaction timestamps to the nearest hour to perform a Left Join with the meteorological data.
    * *Daily Alignment:* Extracting the pure date from transaction timestamps to perform a Left Join with the holiday calendar.

Our goal is to execute these operations immutably, ensuring no records are dropped or artificially duplicated during the joins.

In [33]:
def merge_regdir_bikes(regitered_bikes, direct_bikes):
    """
    Stacks the registered and direct rental tables vertically into a single ledger.
    """
    master_bikes = pd.concat([regitered_bikes, direct_bikes], ignore_index=True)
    return master_bikes

In [34]:
master_bikes = merge_regdir_bikes(df_registered_clean, df_direct_clean)
print("=====REG BIKES: =======")
print(df_registered_clean.head())
print("=====DIR BIKES: =======")
print(df_direct_clean.head())
print("=====MASTER BIKES: =======")
print(master_bikes.head())
print("=====TEST: =======")
len_reg = len(df_registered_clean)
len_dir = len(df_direct_clean)
expected_total = len_reg + len_dir

# Calculate actual merged length
actual_total = len(master_bikes)
print("\n--- Structural Integrity Check ---")
print(f"Registered Rows: {len_reg:,}")
print(f"Direct Rows:     {len_dir:,}")
print(f"Expected Total:  {expected_total:,}")
print(f"Actual Total:    {actual_total:,}")

=====REG BIKES: =======
             datetime  user_id  location_id  is_registered
0 2011-01-01 00:05:09      158           16           True
1 2011-01-01 00:05:21      262           18           True
2 2011-01-01 00:05:39       68           18           True
3 2011-01-01 00:12:05       12            9           True
4 2011-01-01 00:25:58       91           11           True
=====DIR BIKES: =======
             datetime  user_id  location_id  is_registered
0 2011-01-01 00:24:04      232            2          False
1 2011-01-01 00:30:19       54           14          False
2 2011-01-01 00:39:08      201            5          False
3 2011-01-01 01:01:12      298           13          False
4 2011-01-01 01:02:37       23           14          False
=====MASTER BIKES: =======
             datetime  user_id  location_id  is_registered
0 2011-01-01 00:05:09      158           16           True
1 2011-01-01 00:05:21      262           18           True
2 2011-01-01 00:05:39       68          

### 3.1 Days alignment

*To merge our contextual tables, we need to create mathematical "hooks" that perfectly match the granularity of the Weather (hourly) and Holiday (daily) datasets.*
**Here we will floor our time to a precise hour**:


In [35]:
def align_time(master_df):
    """
    Creates perfectly aligned time keys for merging without destroying the precise datetime.
    """
    master_df['join_hour'] = master_df['datetime'].dt.floor('h')
    master_df['join_date'] = pd.to_datetime(master_df['datetime'].dt.date)
    return master_df

In [36]:
timealigned_master_bikes = align_time(master_bikes)
print(timealigned_master_bikes[timealigned_master_bikes['join_hour'].dt.hour > 0].head())
print(len(timealigned_master_bikes))

              datetime  user_id  location_id  is_registered           join_hour  join_date
13 2011-01-01 01:03:49      162            1           True 2011-01-01 01:00:00 2011-01-01
14 2011-01-01 01:04:04        6           20           True 2011-01-01 01:00:00 2011-01-01
15 2011-01-01 01:05:31      166           10           True 2011-01-01 01:00:00 2011-01-01
16 2011-01-01 01:07:41       67           17           True 2011-01-01 01:00:00 2011-01-01
17 2011-01-01 01:09:54       87            1           True 2011-01-01 01:00:00 2011-01-01
3292679


In [37]:
def merge_contextual_data(master_df, weather_df, holidays_df):
    """
    Performs Left Joins to attach weather and holiday context.
    """
    # Merge Weather on the hourly key
    merged_df = pd.merge(
        master_df, 
        weather_df, 
        left_on='join_hour', 
        right_on='datetime', 
        how='left',
        suffixes=('', '_weather')
    )
    
    # Merge Holidays on the daily key
    merged_df = pd.merge(
        merged_df,
        holidays_df,
        left_on='join_date',
        right_on='date',
        how='left'
    )
    
    # Merge puts Nan at the fileds without holidays, we need to substitude it with 'False'
    merged_df['is_holiday'] = merged_df['is_holiday'].fillna(False).astype(bool)
    
    # 4 Drop the redundant merge keys
    columns_to_drop = ['join_hour', 'join_date', 'datetime_weather', 'date']
    merged_df = merged_df.drop(columns=columns_to_drop)
    
    return merged_df

In [38]:
perfect_master_df = merge_contextual_data(timealigned_master_bikes, df_weather_clean, df_holidays_clean)
print("\n=== FINAL MASTER DATASET ===")
print(perfect_master_df.info())
print("\nFirst 3 rows:")
print(perfect_master_df.head(3))


=== FINAL MASTER DATASET ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3292679 entries, 0 to 3292678
Data columns (total 10 columns):
 #   Column                   Dtype         
---  ------                   -----         
 0   datetime                 datetime64[ns]
 1   user_id                  int64         
 2   location_id              int64         
 3   is_registered            bool          
 4   conditions               object        
 5   temperature_c            float64       
 6   perceived_temperature_c  float64       
 7   humidity                 float64       
 8   windspeed_kmh            float64       
 9   is_holiday               bool          
dtypes: bool(2), datetime64[ns](1), float64(4), int64(2), object(1)
memory usage: 207.2+ MB
None

First 3 rows:
             datetime  user_id  location_id  is_registered conditions  temperature_c  perceived_temperature_c  \
0 2011-01-01 00:05:09      158           16           True      clear            3.3        

/var/folders/kn/ts17h9591n904ngr36s97nl00000h0/T/ipykernel_1358/4248770432.py:25: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  merged_df['is_holiday'] = merged_df['is_holiday'].fillna(False).astype(bool)


## Phase 4: Feature Engineering

With our data safely unified into a single master ledger, we must now translate human-readable context into machine-readable mathematical signals. 

**Phase Objectives:**
1. **Temporal Extraction:** We will deconstruct our preserved `datetime` column into individual numerical features (Hour, Month, Day of Week) to allow the model to detect cyclical patterns, such as morning commutes or weekend drops.
2. **Categorical Encoding:** We will translate text-based categories (like weather `conditions`) into numerical formats so the algorithm can process them mathematically.

In [39]:
def engineer_time_features(df):
    """
    Extracts standalone numerical features from the exact timestamp.
    """
    engineered_df = df.copy()
    
    # Extract the hour (0-23)
    # Why: Bike rentals heavily depend on morning/evening rush hours.
    engineered_df['hour'] = engineered_df['datetime'].dt.hour
    
    # Extract the day of the week (Monday=0, Sunday=6)
    # Why: Commuters rent on weekdays; tourists rent on weekends.
    engineered_df['day_of_week'] = engineered_df['datetime'].dt.dayofweek
    
    return engineered_df

In [40]:
time_engin_master = engineer_time_features(perfect_master_df)
print(time_engin_master.head())

             datetime  user_id  location_id  is_registered conditions  temperature_c  perceived_temperature_c  \
0 2011-01-01 00:05:09      158           16           True      clear            3.3                      3.0   
1 2011-01-01 00:05:21      262           18           True      clear            3.3                      3.0   
2 2011-01-01 00:05:39       68           18           True      clear            3.3                      3.0   
3 2011-01-01 00:12:05       12            9           True      clear            3.3                      3.0   
4 2011-01-01 00:25:58       91           11           True      clear            3.3                      3.0   

   humidity  windspeed_kmh  is_holiday  hour  day_of_week  
0      81.0            0.0       False     0            5  
1      81.0            0.0       False     0            5  
2      81.0            0.0       False     0            5  
3      81.0            0.0       False     0            5  
4      81.0          

### 4.1 Condition column mutation

Now, we have to deal with the weather conditions column—it is currently full of text like "clear" or "rain", but firstly we scrutinize the date in the column:

In [41]:
condition_counts = time_engin_master['conditions'].value_counts()

print("--- Unique Weather Conditions ---")
print(condition_counts)

--- Unique Weather Conditions ---
conditions
clear         2338173
clouds         795952
light_rain     158331
heavy_rain        223
Name: count, dtype: int64


### 4.1 Intermediate observation:
* OUR master dataset has 3,292,679 rows. heavy_rain accounts for 0.006% of the data. It is statistically invisible, we will unite this column with rain
* Clear, Cloud and Rain columns are a perfect match for Ture/False statements, we will extract them and add to the our master_df


In [42]:
def encode_weather_conditions(df):
    """
    Consolidates rare weather events and applies One-Hot Encoding to the conditions.
    """
    engineered_df = df.copy()
    engineered_df['conditions'] = engineered_df['conditions'].replace(
        ['heavy_rain', 'light_rain'], 
        'rain'
    )
    
    # One-Hot Encoding: Explode the text column into independent boolean columns
    # prefix='weather' ensures ther new columns are neatly named (e.g., 'weather_clear')
    # dtype=bool ensures we get True/False instead of 1/0, matching is_holiday column
    engineered_df = pd.get_dummies(
        engineered_df, 
        columns=['conditions'], 
        prefix='weather',
        dtype=bool
    )
    
    return engineered_df

In [43]:
print("Encoding categorical weather data...")
weathertime_engin_master = encode_weather_conditions(time_engin_master)

print("\n--- Encoded Weather Columns ---")
print(weathertime_engin_master[['datetime', 'weather_clear', 'weather_clouds', 'weather_rain']].head(5))

Encoding categorical weather data...

--- Encoded Weather Columns ---
             datetime  weather_clear  weather_clouds  weather_rain
0 2011-01-01 00:05:09           True           False         False
1 2011-01-01 00:05:21           True           False         False
2 2011-01-01 00:05:39           True           False         False
3 2011-01-01 00:12:05           True           False         False
4 2011-01-01 00:25:58           True           False         False


## Phase 5: Pre-Processing Summary

In this final phase of data preparation, we translated human-readable context into machine-readable mathematical signals. Machine learning algorithms require numerical or boolean inputs to calculate weights and identify patterns. Our primary objective was to extract the maximum predictive value from our raw data without introducing statistical noise or false mathematical hierarchies.

### Summary of Engineered Features

**1. Temporal Feature Extraction**
* **Action:** Deconstructed the preserved `datetime` column into three discrete numerical features: `hour` (0-23), `day_of_week` (0-6), and `month` (1-12).
* **Rationale:** Bike rental demand is heavily dictated by human cyclical behavior. A predictive model cannot natively parse a raw timestamp string to understand that "8:00 AM on a Monday" represents a commuter rush, or that "July" represents peak summer usage. Extracting these specific integers allows the algorithm to detect and weigh these recurring temporal patterns.

**2. Categorical Consolidation (Dimensionality Reduction)**
* **Action:** Merged the extremely rare `heavy_rain` occurrences (223 rows) with `light_rain` to create a single, unified `rain` category.
* **Rationale:** Statistical significance. An algorithm cannot extract reliable mathematical rules from an event that occurs in 0.006% of a 3.2-million-row dataset. Leaving it as an independent feature would only introduce noise and slow down model convergence.

**3. One-Hot Encoding**
* **Action:** Converted the consolidated `conditions` text column into independent boolean flags (`weather_clear`, `weather_clouds`, `weather_rain`) and dropped the original text column.
* **Rationale:** Avoiding ordinal traps. Converting text to an enumerated list (e.g., clear=1, clouds=2, rain=3) forces the algorithm to mathematically assume that rain is "three times greater" than clear weather. Exploding the categories into independent True/False columns allows the model to evaluate the impact of each weather state objectively and independently.

---
**Pipeline Status:** The master dataset is fully unified, structurally normalized, and mathematically sanitized. The exploratory analysis and feature engineering phases are complete. The modular transformation functions are now ready to be migrated into a formal Dagster orchestration pipeline.